# Multi-Config Parallel RAG Evaluation

This notebook:

1. **Loads the dataset once** (raw samples + parsed documents are built a single time).
2. **Loads multiple RAG configs at once** from their config files.
3. **Runs every config in its own pipeline, in parallel**, reusing the shared documents.
4. **Generates a report** with the pluggable `ReportGenerator` + `ReportStrategy`
   classes (one strategy: `DetailedQueryReportStrategy`).

The report contains, per config:
- a per-query table (query, retrieved documents as an array **with all scores**,
  generated answer, and every TRACe score next to its ground truth + per-query deviation);
- an aggregate table with the **mean score, mean ground truth, standard deviation,
  and mean absolute error** for every relevant TRACe score.

## 1. Setup & Dependencies

In [1]:
get_ipython().system('pip install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk pandas -q')

zsh:1: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/bin/pip: bad interpreter: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/real-world-rag-system/venv/bin/python: no such file or directory

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /Users/adityanarayan/.pyenv/versions/3.11.9/bin/python -m pip install --upgrade pip


## 2. Imports

In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from dotenv import load_dotenv

from rag.config.enums import Mode
from rag.config.loader import ConfigLoader
from rag.pipeline.rag_pipeline import RAGPipeline
from reporting import (
    DetailedQueryReportStrategy,
    ReportGenerator,
)
from rag.models.pipeline_run_result import PipelineRunResult
from rag.models.query_result import QueryResult
from ingestion import (
    DataProcessor,
    DatasetLoadingConfig,
    HuggingFaceLoader,
    ParserFactory,
    ParserType,
)

load_dotenv(override=True)
print("HuggingFace token loaded:", bool(os.getenv("HF_TOKEN")))
print("Groq API key loaded:", bool(os.getenv("GROQ_API_KEY")))

HuggingFace token loaded: True
Groq API key loaded: True


## 3. Load the data ONCE

The dataset is loaded and parsed a single time. The resulting `documents` list
is shared by every pipeline below, and `eval_items` holds the queries together
with their dataset ground-truth TRACe scores.

In [ ]:
# Keep the demo small and fast; raise these to scale up.
NUM_SAMPLES_FOR_INDEX = 10   # samples whose documents are indexed
NUM_QUERIES = 3              # queries (and ground truths) to evaluate

loader = HuggingFaceLoader(
    dataset_name="galileo-ai/ragbench",
    subset="covidqa",
    split="test",
    config=DatasetLoadingConfig(limit=NUM_SAMPLES_FOR_INDEX),
)
raw_data = loader.load()
print(f"Loaded {len(raw_data)} raw samples")

parser = ParserFactory.create_parser(ParserType.TITLE_PASSAGE)
processor = DataProcessor(parser_strategy=parser)
documents = processor.process_dataset(raw_data)
print(f"Parsed {len(documents)} documents (built once, shared by all configs)")

eval_items = []
for sample in raw_data[:NUM_QUERIES]:
    eval_items.append({
        "query": sample["question"],
        "ground_truth": {
            "relevance_score": sample.get("relevance_score", 0.0),
            "utilization_score": sample.get("utilization_score", 0.0),
            "completeness_score": sample.get("completeness_score", 0.0),
            "adherence_score": sample.get("adherence_score", False),
        },
    })
print(f"Prepared {len(eval_items)} queries with ground truth")

Loading RAGBench dataset: covidqa (test)...
Loaded 10 samples
Loaded 10 raw samples
Parsed 40 documents (built once, shared by all configs)
Prepared 3 queries with ground truth


## 4. Load multiple configs at once

Each config file describes a full pipeline (chunking, embedding, retrieval,
generation, evaluation). They are loaded into memory together; `Mode.TEST`
keeps logging quiet while pipelines run concurrently.

In [ ]:
CONFIG_PATHS = {
    "high_quality_large": "experiment_configs/high_quality_large.yaml",
    "high_quality_small": "experiment_configs/high_quality_small.yaml"
}

configs = {}
for name, path in CONFIG_PATHS.items():
    cfg = ConfigLoader.load(path)
    cfg.mode = Mode.TEST  # quiet logs during parallel execution
    configs[name] = cfg
    print(
        f"{name:>13}: chunking={cfg.chunking.type.value}, "
        f"retrieval={cfg.retrieval.type.value}, "
        f"gen_model={cfg.generation.config.model}"
    )

   fast_local: chunking=fixed_window, retrieval=dense, gen_model=llama-3.1-8b-instant


## 5. Per-config pipeline runner

`run_config` builds one pipeline from a config, indexes the **shared** documents,
runs every query, and packages the output (retrieved docs with scores, answer,
predicted scores, ground truth) into a `PipelineRunResult` — the input the
report generator expects.

In [10]:
from rag.pipeline.rag_pipeline import RAGPipeline
from rag.models.pipeline_run_result import PipelineRunResult
from rag.models.query_result import QueryResult


def run_config(config, documents, raw_data, num_report_queries = 5):
    """
    Run a single RAG configuration end-to-end and return a PipelineRunResult.

    Args:
        config: Loaded RAGConfig
        documents: Parsed documents (shared across all experiments)
        raw_data: Raw dataset samples
        num_report_queries: Number of queries to evaluate

    Returns:
        PipelineRunResult
    """

    pipeline = RAGPipeline(config)

    # Uses cache automatically if available.
    pipeline.build_index(documents)

    records = []

    for sample in raw_data[:num_report_queries]:

        query = sample["question"]

        result = pipeline.query(query)

        records.append(
            QueryResult(
                query=query,
                retrieved_docs=result["retrieved_docs"],
                answer=result["response"],
                metadata={
                    "predicted_scores": result["scores"],
                    "ground_truth_scores": {
                        "relevance_score": sample.get("relevance_score", 0.0),
                        "utilization_score": sample.get("utilization_score", 0.0),
                        "completeness_score": sample.get("completeness_score", 0.0),
                        "adherence_score": sample.get("adherence_score", False),
                    },
                },
            )
        )

    return PipelineRunResult(
        records=records,
        config=config,
    )

## 6. Run all configs in parallel

Each config runs in its own pipeline on a separate thread. Because retrieval is
followed by LLM generation + TRACe evaluation (I/O-bound API calls), running the
configs concurrently overlaps that latency.

In [12]:
runs = []

run = run_config(
    next(iter(configs.values())),
    documents,
    raw_data,
)
runs.append(run)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16292.55it/s]


In [14]:
from concurrent.futures import ThreadPoolExecutor, as_completed

runs = []

with ThreadPoolExecutor(max_workers=len(configs)) as executor:

    futures = {
        executor.submit(
            run_config,
            config,
            documents,
            raw_data,
        ): config.name
        for config in configs.values()
    }

    for future in as_completed(futures):
        run = future.result()
        runs.append(run)
        print(f"Finished: {run.config.name}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9514.04it/s]


Finished: fast-local


## 7. Generate the report

`ReportGenerator` is a thin orchestrator holding a registry of report
strategies. Here we register the single available strategy,
`DetailedQueryReportStrategy`, and generate the report from the parallel runs.
New report formats can be added by implementing `ReportStrategy` and registering
it — no change to `ReportGenerator` required.

In [15]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

generator = ReportGenerator(DetailedQueryReportStrategy())
print("Available report strategies:", generator.strategies)

for run in runs:
    report = generator.generate(run)
    report.save_json(f"reports/{run.config.name}_report.json")
    report.display()

Available report strategies: ['detailed_query']


# RAG Multi-Config Evaluation Report

_Strategy: detailed_query_

## Config: `fast-local`

**name**: fast-local  •  **mode**: test  •  **providers**: {'groq': {'type': 'groq', 'api_key_env': 'GROQ_API_KEY', 'params': {'cooldown_seconds': 60}}}  •  **chunking**: {'type': 'fixed_window', 'config': {'window_size': 256, 'overlap': 50}}  •  **embedding**: {'type': 'sentence_transformer', 'config': {'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'model': None, 'dimension': 384}}  •  **vector_store**: {'type': 'faiss', 'config': {'dimension': 384}}  •  **retrieval**: {'type': 'dense', 'config': {'top_k': 5}}  •  **reranker**: None  •  **generation**: {'strategy': 'default', 'provider': 'groq', 'config': {'model': 'llama-3.1-8b-instant', 'max_tokens': 512, 'temperature': 0.5, 'system_prompt': None}}  •  **evaluation**: {'type': 'trace', 'provider': 'groq', 'config': {'model': 'llama-3.1-8b-instant', 'max_tokens': 2000, 'temperature': 0.0}}  •  **cache**: {'enabled': True, 'cache_dir': './cache'}

### Per-query results

,query,retrieved_documents,answer,relevance_score__pred,relevance_score__gt,relevance_score__deviation,utilization_score__pred,utilization_score__gt,utilization_score__deviation,completeness_score__pred,completeness_score__gt,completeness_score__deviation,adherence_score__pred,adherence_score__gt,adherence_score__deviation
0,Which viruses may not cause prolonged inflammation due to strong induction o...,[{'text': 'successful treatment for HCV serves to circumvent the viral inhib...,"Based on the context, the viruses that may not cause prolonged inflammation ...",0.9091,0.4118,0.4973,0.2727,0.1765,0.0962,0.3000,0.4286,-0.1286,0.0,0.0,0.0
1,When was the first case of COVID-19 identified?,"[{'text': 'Text: In the WHO European Region, COVID-19 surveillance was imple...",The first case of COVID-19 identified is not explicitly mentioned in the pro...,0.2143,0.2692,-0.0549,0.3571,0.0769,0.2802,1.0000,0.2857,0.7143,0.0,1.0,-1.0
2,How many antigens could be detected by Liew's multiplex ELISA test?,[{'text': 'Antibody arrays for simultaneous multiple antigen quantification ...,"According to Document 1, Liew validated one multiplex ELISA for the detectio...",1.0000,0.1250,0.8750,0.1250,0.0625,0.0625,0.1250,0.5000,-0.3750,1.0,1.0,0.0
3,What is the structure of Hantaan virus?,"[{'text': 'The diameter of hantavirus particles is 80-US210 nm, and the stru...","According to Document 1, the structure of Hantaan virus (hantavirus) is eith...",0.9167,0.3000,0.6167,0.0833,0.3000,-0.2167,0.0909,1.0000,-0.9091,1.0,1.0,0.0
4,How many people did SARS-CoV infect?,[{'text': 'countries between 7 and 43 million individuals have been infected...,"According to Document 4, SARS-CoV caused 8098 reported cases and 774 deaths ...",1.0000,0.0667,0.9333,0.6000,0.0667,0.5333,0.6000,1.0000,-0.4000,0.0,1.0,-1.0


### Aggregate TRACe scores (mean / ground truth / deviation)

,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,relevance_score,0.8080,0.2345,0.2994,0.1242,0.5955
1,utilization_score,0.2876,0.1365,0.1848,0.0919,0.2378
2,completeness_score,0.4232,0.6429,0.3401,0.2997,0.5054
3,adherence_score,0.4000,0.8000,0.4899,0.4000,0.4000
